# scPRINT-2 Dataset Composition

This notebook generates summary figures for the scPRINT-2 dataset, including tissue, organism, assay, cell-type, ethnicity, disease, and simplified-age distributions.

In [1]:
! pip install plotly "anywidget>=0.9.13" kaleido

error: externally-managed-environment

× This environment is externally managed
╰─> This Python installation is managed by uv and should not be modified.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detailed specification.


In [2]:
import plotly.express as px
import plotly.io as pio
import pandas as pd 
import numpy as np
import json
from pathlib import Path

### Data

In [3]:
COUNTS_CANDIDATES = (
    Path("scprint_v2_composition_counts.json"),
    Path("figures/scprint_v2_composition_counts.json"),
)
COUNTS_PATH = next((path for path in COUNTS_CANDIDATES if path.exists()), None)
if COUNTS_PATH is None:
    raise FileNotFoundError(
        "Could not find scprint_v2_composition_counts.json next to the notebook "
        "or under figures/."
    )

with COUNTS_PATH.open() as handle:
    composition_counts = json.load(handle)

TOTAL_CELLS = composition_counts["total_cells"]


def collapse_counts(field, top_n=15):
    """Keep the most abundant labels and combine the remainder as other."""
    counts = composition_counts["fields"][field]["counts"]
    ordered = sorted(counts.items(), key=lambda item: item[1], reverse=True)
    if top_n is None or len(ordered) <= top_n:
        return dict(ordered)
    retained = dict(ordered[:top_n])
    retained[f"other ({len(ordered) - top_n} categories)"] = sum(
        count for _, count in ordered[top_n:]
    )
    return retained


# Refresh the two existing composition charts from the same live collection.
species = collapse_counts("organism", top_n=None)
tissues = collapse_counts("tissue", top_n=15)

assays = collapse_counts("assay", top_n=9)

### Tissue Distribution

With all labels:

In [4]:
# Create a camembert plot
fig = px.pie(
    names=list(tissues.keys()),
    values=tissues.values(),
    title="Tissues in scPRINT-2",
    hole=0.3  # for a donut-style camembert
)

fig.show()
# Export to PDF
# pio.write_image(fig, "tissues_pie_plotly.pdf")

With no labels:

In [5]:
# Create a camembert plot
fig = px.pie(
    names=list(tissues.keys()),
    values=tissues.values(),
    title="Tissues in scPRINT-2",
    hole=0.3  # for a donut-style camembert
)

fig.update_traces(textinfo="none") 
fig.show()
# Export to PDF
# pio.write_image(fig, "tissues_pie_plotly.pdf")

labels = list(fig.data[0].labels)
values = list(fig.data[0].values)
total = sum(values)

df_pct = pd.DataFrame({
    "label": labels,
    "count": values,
    "percent": [v / total * 100 for v in values],
}).sort_values("percent", ascending=False)
df_pct

,label,count,percent
15,other (311 categories),87104297,25.061807
0,lung,42868831,12.334298
1,blood,34179961,9.834321
2,brain,29530595,8.496597
3,colon,25830811,7.432088
4,unknown,23810521,6.850806
5,pancreas,21597602,6.214101
6,embryo,16976623,4.884545
7,skin of body,12513695,3.600463
8,liver,10683313,3.073822


With top 3 labels:

In [6]:
labels = list(tissues.keys())
values = np.array(list(tissues.values()), dtype=float)

fig = px.pie(
    names=labels,
    values=values,
    title="Tissues in scPRINT-2",
    hole=0.3,
)

# indices des 3 plus grandes parts
top3_idx = set(np.argsort(values)[-3:])

# texte seulement pour top3 (label + %)
custom_text = []
for i, (lab, val) in enumerate(zip(labels, values)):
    if i in top3_idx:
        pct = 100 * val / values.sum()
        custom_text.append(f"{lab}<br>{pct:.2f}%")
    else:
        custom_text.append("")

fig.update_traces(
    text=custom_text,
    texttemplate="%{text}",
    textinfo="none",   # on utilise uniquement le texte custom
)

fig.show()

### Specie distribution

With all labels:

In [7]:
# Create a camembert plot
fig = px.pie(
    names=list(species.keys()),
    values=species.values(),
    title="Species in scPRINT-2",
    hole=0.3,  # for a donut-style camembert
)
fig.show()
# Export to PDF
# pio.write_image(fig, "species_pie_plotly.pdf")

With no labels: 

In [8]:
# Create a camembert plot
fig = px.pie(
    names=list(species.keys()),
    values=species.values(),
    title="Species in scPRINT-2",
    hole=0.3,  # for a donut-style camembert
)
fig.update_traces(textinfo="none")
fig.show()
# Export to PDF
# pio.write_image(fig, "species_pie_plotly.pdf")

labels = list(fig.data[0].labels)
values = list(fig.data[0].values)
total = sum(values)

df_pct = pd.DataFrame({
    "label": labels,
    "count": values,
    "percent": [v / total * 100 for v in values],
}).sort_values("percent", ascending=False)
df_pct

,label,count,percent
0,human,230724984,66.384613
1,mouse,102753838,29.564521
2,macaque,3161179,0.909540
3,zebrafish,3117237,0.896897
4,pig,1641179,0.472203
5,arabidopsis thaliana,1564711,0.450201
6,drosophila melanogaster (fruit fly),1564460,0.450129
7,chicken,606680,0.174555
8,naked mole-rat female,441042,0.126897
9,rabbit,417091,0.120006


With top 3 labels: 

In [9]:
labels = list(species.keys())
values = np.array(list(species.values()), dtype=float)

# Create a camembert plot
fig = px.pie(
    names=list(species.keys()),
    values=species.values(),
    title="Species in scPRINT-2",
    hole=0.3,  # for a donut-style camembert
)
# indices des 3 plus grandes parts
top3_idx = set(np.argsort(values)[-3:])

# texte seulement pour top3 (label + %)
custom_text = []
for i, (lab, val) in enumerate(zip(labels, values)):
    if i in top3_idx:
        pct = 100 * val / values.sum()
        custom_text.append(f"{lab}<br>{pct:.2f}%")
    else:
        custom_text.append("")

fig.update_traces(
    text=custom_text,
    texttemplate="%{text}",
    textinfo="none",   # on utilise uniquement le texte custom
)

fig.show()

### Assay distribution

With all labels:

In [10]:
# Create a camembert plot
fig = px.pie(
    names=list(assays.keys()),
    values=assays.values(),
    title="Assays in scPRINT-2",
    hole=0.3,  # for a donut-style camembert
)
fig.show()
# Export to PDF
# pio.write_image(fig, "assays_pie_plotly.pdf")

With no labels:

In [11]:
# Create a camembert plot
fig = px.pie(
    names=list(assays.keys()),
    values=assays.values(),
    title="Assays in scPRINT-2",
    hole=0.3,  # for a donut-style camembert
)
fig.show()

labels = list(fig.data[0].labels)
values = list(fig.data[0].values)
total = sum(values)

df_pct = pd.DataFrame({
    "label": labels,
    "count": values,
    "percent": [v / total * 100 for v in values],
}).sort_values("percent", ascending=False)
df_pct

,label,count,percent
0,10x 3' transcription profiling,146909525,42.269076
1,10x 3' v3,121220624,34.877819
2,10x 5' transcription profiling,28919386,8.320738
3,sci-RNA-seq,14924076,4.293982
4,10x 3' v2,12021763,3.458924
9,other (23 categories),6994945,2.012598
5,10x multiome,6890747,1.982618
6,10x 5' v1,4705177,1.353782
7,10x 5' v2,3104399,0.893203
8,Slide-seqV2,1867286,0.537259


With top 3 labels:

In [12]:
labels = list(assays.keys())
values = np.array(list(assays.values()), dtype=float)

fig = px.pie(
    names=labels,
    values=values,
    title="Assays in scPRINT-2",
    hole=0.3,
)

# indices des 3 plus grandes parts
top3_idx = set(np.argsort(values)[-3:])

# texte seulement pour top3 (label + %)
custom_text = []
for i, (lab, val) in enumerate(zip(labels, values)):
    if i in top3_idx:
        pct = 100 * val / values.sum()
        custom_text.append(f"{lab}<br>{pct:.2f}%")
    else:
        custom_text.append("")

fig.update_traces(
    text=custom_text,
    texttemplate="%{text}",
    textinfo="none",   # on utilise uniquement le texte custom
)

fig.show()

In [13]:
import matplotlib.pyplot as plt

# Create a pie chart
plt.figure(figsize=(10, 10))
plt.pie(
    tissues.values(),
    labels=tissues.keys(),
    autopct='%1.1f%%',
    pctdistance=0.85,
    labeldistance=1.1
)

# Add a circle at the center to make it a donut chart
centre_circle = plt.Circle((0,0), 0.70, fc='white')
fig = plt.gcf()
fig.gca().add_artist(centre_circle)

plt.title("Tissues in scPRINT-2")
plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle

# Save as PDF
plt.savefig('tissues_pie.pdf', format='pdf', bbox_inches='tight')
plt.close()

## Requested metadata distributions

These charts use the live LaminDB collection
`scPRINT-V2 (all+tahoe+scbase) filtered`: **347,557,928 cells** across
**26,332 artifacts**, counted on 2026-08-18. Ontology IDs are
resolved through the connected Bionty registries. Each pie keeps the 15 most
abundant labels and combines the remainder into `other`; the three largest
slices are labeled on-chart. Simplified age is the training column `age_group`.

In [14]:
def plot_composition_pie(field, title, top_n=15, top_labels=3):
    counts = collapse_counts(field, top_n=top_n)
    summary = pd.DataFrame(
        {"label": counts.keys(), "count": counts.values()}
    ).sort_values("count", ascending=False, ignore_index=True)
    summary["percent"] = summary["count"] / summary["count"].sum() * 100

    fig = px.pie(
        summary,
        names="label",
        values="count",
        title=f"{title} in scPRINT-2",
        hole=0.3,
    )
    shown = set(summary.nlargest(top_labels, "count").index)
    custom_text = [
        f"{row.label}<br>{row.percent:.2f}%" if index in shown else ""
        for index, row in summary.iterrows()
    ]
    fig.update_traces(
        text=custom_text,
        texttemplate="%{text}",
        textinfo="none",
        hovertemplate=(
            "%{label}<br>%{value:,} cells<br>%{percent}<extra></extra>"
        ),
    )
    fig.show()
    return summary

### Cell-type distribution

In [15]:
cell_type_summary = plot_composition_pie("cell_type", "Cell types")
cell_type_summary

,label,count,percent
0,unknown,284738754,81.925553
1,other (788 categories),39628555,11.402000
2,neuron,3831426,1.102385
3,glutamatergic neuron,2599351,0.747890
4,neural cell,2063265,0.593646
5,L2/3-6 intratelencephalic projecting glutamate...,1894072,0.544966
6,fibroblast,1587992,0.456900
7,mesodermal cell,1539949,0.443077
8,"CD4-positive, alpha-beta T cell",1534477,0.441503
9,oligodendrocyte,1514149,0.435654


### Tissue distribution (refreshed)

In [16]:
tissue_summary = plot_composition_pie("tissue", "Tissues")
tissue_summary

,label,count,percent
0,other (311 categories),87104297,25.061807
1,lung,42868831,12.334298
2,blood,34179961,9.834321
3,brain,29530595,8.496597
4,colon,25830811,7.432088
5,unknown,23810521,6.850806
6,pancreas,21597602,6.214101
7,embryo,16976623,4.884545
8,skin of body,12513695,3.600463
9,liver,10683313,3.073822


### Self-reported ethnicity distribution

In [17]:
ethnicity_summary = plot_composition_pie("ethnicity", "Self-reported ethnicities")
ethnicity_summary

,label,count,percent
0,unknown,232103344,66.781197
1,European,103169011,29.683976
2,East Asian,5224564,1.503221
3,African,2076086,0.597335
4,Asian,1057707,0.304325
5,African American,849525,0.244427
6,Hispanic or Latin American,686359,0.197480
7,Han Chinese,580297,0.166964
8,other (16 categories),392580,0.112954
9,Korean,366741,0.105519


### Disease distribution

In [18]:
disease_summary = plot_composition_pie("disease", "Diseases")
disease_summary

,label,count,percent
0,unknown,138300318,39.792019
1,normal,55942291,16.095818
2,other (419 categories),51810259,14.906942
3,adenocarcinoma,36356863,10.460663
4,carcinoma,11976298,3.445842
5,breast ductal adenocarcinoma,9080575,2.612680
6,non-small cell lung carcinoma,8480880,2.440134
7,melanoma,7076568,2.036083
8,large cell carcinoma,6037420,1.737097
9,COVID-19,5533607,1.592139


### Simplified-age distribution

In [19]:
simplified_age_summary = plot_composition_pie("simplified_age", "Simplified age groups")
simplified_age_summary

,label,count,percent
0,unknown,195100156,56.134572
1,adult stage,65403174,18.817920
2,60-79 year-old stage,34649390,9.969386
3,young adult stage,12011423,3.455949
4,fetal stage,9596980,2.761261
5,organogenesis stage,7570516,2.178203
6,middle aged stage,6619942,1.904702
7,late adult stage,5115316,1.471788
8,juvenile stage (5-14 yo),4064903,1.169561
9,80 year-old and over stage,2638406,0.759127
